# RSA: ColBERT + MUVERA first baselines

Choose **Runtime → Change runtime type → GPU**, then run all cells.
This compares pretrained ColBERTv2 and a MUVERA reference/Faiss HNSW baseline
on the same held-out fashion products. The binary/FP32 semantic heads use
fashion teacher supervision; ColBERT does not. Results are exploratory.

The notebook saves prepared inputs and results to Drive. Rerunning the same
configuration reuses verified embeddings. Change the run directory after a code
or configuration change. See `docs/LATE_INTERACTION.md` for metric definitions.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/ras-late-interaction')
BRANCH = 'codex/colbert-muvera-baselines'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hanialshater/ras.git', str(REPO)], check=True)
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                str(REPO) + '[dev,benchmark,late-interaction]'], check=True)


In [ ]:
import os
os.chdir(REPO)
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':' + str(REPO)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_late_interaction.py'], check=True)


## Initial experiment

Start with 30 queries and one split; use 200 queries and seeds 7/17/27 after
inspecting the initial results. Reuse the same parameters when resuming.
The GPU encodes titles/images/queries; reference retrieval and MaxSim run on CPU.


In [ ]:
RUN = Path('/content/drive/MyDrive/ras_late_interaction_seed7_v1')
command = [sys.executable, '-u', '-m', 'experiments.late_interaction_baselines',
           '--output-dir', str(RUN), '--queries', '30', '--seed', '7',
           '--fde-seed', '7', '--k', '50', '--candidates', '100', '500', '1000',
           '--repetitions', '8', '--partition-bits', '4', '--fde-dim', '4096',
           '--backend', 'hnsw']
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import pandas as pd
summary = pd.read_csv(RUN / 'summary.csv')
display(summary[['scope', 'method', 'candidate_budget', 'queries',
                 'recall_mean', 'precision_at_k_mean', 'ndcg_mean',
                 'fill_rate_mean', 'colbert_topk_recall_mean']])
print('Keep shared-pool recall separate from full-corpus recall.')
print('ColBERT top-K recall measures approximation, not semantic relevance.')
print('Timing and memory scope:', RUN / 'scope.json', RUN / 'memory.json')


In [ ]:
from google.colab import files
files.download(str(RUN / 'late_interaction_results.zip'))
